# Scrye — Persona Factory (demographic conditioning & post-stratification)

A **persona factory** turns any SimBench segment (country and/or demographic
attributes) into a conditioning prompt, and uses each subset's demographic
distribution to **recombine** subgroup predictions into coarser estimates. Two
levers, both swappable systems in the `experiment` harness:

1. **Conditioning style** — a registry of `PromptStrategy`s (`scrye.persona`).
   `simbench_faithful` is the paper's first-person persona; the others are
   third-person / distributional reframings and a stronger persona.
2. **Post-stratification** — `PostStratificationPredictor` decomposes a coarse
   target into finer cells (weights from `group_size`), predicts each, and
   averages by population share (`scrye.distributions`).

**The tension we're testing.** SimBench found naive demographic conditioning
*degrades* scores — but that was first-person personas, many weak models, one
prompt. Stronger models + better framings, or exploiting population structure
via post-stratification, might do better. We **measure** rather than assume.

**Holdout discipline** (same as `01_experiments.ipynb`): iterate on **dev**,
select on **val**, score **test** exactly once. Everything below runs on dev.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from scrye.config import DEFAULT_MODEL, OPENROUTER_MODELS
from scrye.data import load_all
from scrye.splits import make_split
from scrye import evaluate as ev
from scrye import experiment as xp
from scrye import viz
from scrye.persona import STRATEGIES, COUNTRIES, ALL_COUNTRIES, get_strategy, verbalize_segment
from scrye.distributions import SegmentWeights, distribution_table

recs = load_all()
all_records = recs["pop"] + recs["grouped"]
grouped = recs["grouped"]

# Normalizers (SimBench Eq. 2) from the FULL data so scores stay comparable.
normalizers = ev.build_normalizers(all_records)
# Post-stratification weights, derived once from group_size counts.
weights = SegmentWeights(grouped)
# Leave-family-out split (deterministic).
split = make_split(all_records, seed=0)

print(f"{len(all_records)} records | {len(ALL_COUNTRIES)} distinct countries | "
      f"strategies: {sorted(STRATEGIES)}")
split.summary()

## 1. The country catalog

All countries from the SimBench grouped split, by source dataset.

In [ ]:
for ds, cs in COUNTRIES.items():
    head = ", ".join(cs[:6]) + (" …" if len(cs) > 6 else "")
    print(f"{ds:16s} {len(cs):3d}  {head}")

## 2. The factory — render every conditioning style

Same segment, five framings. `simbench_faithful` reproduces the benchmark's
first-person persona verbatim (the baseline the "conditioning hurts" result was
measured on); the rest are the hypothesized improvements.

In [ ]:
# A rich two-variable example: country × attribute.
example = next(r for r in grouped
               if len(r.segment) == 2 and any(k in r.segment for k in ("cntry", "country")))
print("segment      :", example.segment)
print("group_prompt :", repr(example.group_prompt))
print("descriptor   :", verbalize_segment(example.segment))
print()
for name in sorted(STRATEGIES):
    print(f"================= {name} =================")
    for m in get_strategy(name).build_messages(example):
        print(f"[{m['role']}]\n{m['content']}\n")

## 3. Demographic distribution tables (post-stratification weights)

`distribution_table` normalizes `group_size` within every
(dataset, question, grouping-keys) subset, giving each cell's population share —
the weights used to combine inferences across groups.

In [ ]:
rows = distribution_table(grouped)
dist = pd.DataFrame([
    {"dataset": r.dataset, "grouping_keys": r.grouping_keys,
     **r.segment, "group_size": r.group_size, "weight": round(r.weight, 3)}
    for r in rows
])
print(f"{len(dist)} cells across {dist['grouping_keys'].nunique()} subset types")

# Example: the gender split within one country for one question.
mask = dist["grouping_keys"].apply(lambda k: k == ("country", "gender"))
dist[mask].head(8)

### Validation — the recombination identity

If the weights are right, the `group_size`-weighted average of the
attribute-subgroup **truths** should reproduce the country-marginal **truth**.
This is the offline backbone for post-stratification (no LLM involved).

In [ ]:
errs = []
for rec in grouped:
    if rec.grouping_keys != ("country",):
        continue
    children = weights.children(rec, over="gender")
    if len(children) < 2:
        continue
    comb = {o: sum(w * c.human_answer.get(o, 0.0) for w, c in children) for o in rec.options}
    errs.append(0.5 * sum(abs(comb[o] - rec.human_answer.get(o, 0.0)) for o in rec.options))

print(f"recombination TVD vs marginal over {len(errs)} country-questions: "
      f"mean={np.mean(errs):.4f}  median={np.median(errs):.4f}  max={np.max(errs):.4f}")

## 4. Experiment A — which conditioning style scores best?  ← live, dev only

Hold the model fixed, vary only the conditioning style, on a dataset-stratified
subsample of the **conditioned** dev records (style only matters where there's a
segment). Each style is a registered simulation system, so this is the same
`xp.compare` machinery as `01_experiments.ipynb`.

> Spends a little API budget on first run (responses cache to disk; re-runs are free).

In [ ]:
MODEL = "gemini-flash-lite"     # the swappable model variable
N_DEV = 120                     # bump for tighter CIs

grouped_dev = [r for r in split.dev if not r.is_population]
dev_eval = ev.stratified_sample(grouped_dev, n=N_DEV, seed=0)
print(f"{len(dev_eval)} conditioned dev records across "
      f"{len({r.dataset_name for r in dev_eval})} datasets")

SYSTEMS = {"uniform (floor)": xp.build_pipeline(predictor="uniform")}
for style in ["simbench_faithful", "representative_sample", "persona_embodiment",
              "anti_flattening", "contextualized"]:
    SYSTEMS[style] = xp.build_pipeline(model=MODEL, predictor=style)

topline = xp.compare(SYSTEMS, dev_eval, normalizers=normalizers)
topline

In [ ]:
viz.plot_system_comparison(topline)

## 5. Experiment B — post-stratification vs direct, on country marginals  ← live

The combiner test. Take country-marginal records that admit a decomposition,
and compare three systems scored against the **marginal** truth:

- **uniform** (floor),
- **direct** — condition on the country and predict the marginal in one shot,
- **post-stratified** — predict each subgroup cell and recombine by population
  weight.

If decompose-then-recombine exploits real population structure, it beats direct
on these targets.

In [ ]:
# Country marginals (in dev) that admit a >=2-cell decomposition.
marginals = [r for r in split.dev
             if r.grouping_keys == ("country",) and len(weights.children(r)) >= 2]
marg_eval = ev.stratified_sample(marginals, n=80, seed=0)
print(f"{len(marg_eval)} decomposable country marginals")

BASE_STYLE = "simbench_faithful"
SYSTEMS_B = {
    "uniform (floor)":        xp.build_pipeline(predictor="uniform"),
    f"direct ({BASE_STYLE})": xp.build_pipeline(model=MODEL, predictor=BASE_STYLE),
    "post-stratified":        xp.post_strat_pipeline(weights, model=MODEL, strategy=BASE_STYLE),
}
toplineB = xp.compare(SYSTEMS_B, marg_eval, normalizers=normalizers)
toplineB

In [ ]:
post = SYSTEMS_B["post-stratified"].predictor
print(f"post-strat: decomposed {post.n_decomposed} records, fell back on {post.n_fallback}")
viz.plot_system_comparison(toplineB)

## 6. Drill-in & discipline reminder

`compare()` stashes each system's per-record frame in `.attrs["results"]`, so the
whole `viz` suite works on any system — e.g. the mode-seeking test below.

- Everything above ran on **dev** — iterate freely.
- Use **val** to pick the winning style / system and fit any calibration.
- Score **test** once, at the end, for reported numbers.
- Cost note: post-stratification makes ~one LLM call **per subgroup cell**, so it
  is several× the API cost of direct conditioning.

In [ ]:
label = "post-stratified"
res = toplineB.attrs["results"][label]
print("per-record rows:", len(res))
viz.plot_score_vs_entropy(res)